# Guide 3 — Linked records: instance, test, dataset, and publishing

This guide builds the full provenance chain — CellSpecification → CellInstance → TestSpec → Test → Dataset — using the `Workspace` API, validates it, and publishes it.

**Estimated time:** 30 minutes
**Prerequisites:** [Guide 2 — First cell spec](02-first-cell-type.ipynb)

Everything writes under `.battinfo/notebooks/guide-03`.

In [ ]:
import os, json, warnings
from pathlib import Path

_here = Path.cwd()
if (_here / "src" / "battinfo").exists():
    REPO = _here
elif (_here.parent.parent / "src" / "battinfo").exists():
    REPO = _here.parent.parent
    os.chdir(REPO)
else:
    REPO = _here

print(f"Repo root: {REPO}")

WORKSPACE = REPO / '.battinfo/notebooks/guide-03'
WORKSPACE.mkdir(parents=True, exist_ok=True)
print('Workspace:', WORKSPACE)

## The Workspace

`Workspace` is BattINFO's Python authoring surface for linked records. It keeps all objects in memory, assigns consistent IRIs, and writes everything to disk in one call.

In [ ]:
from battinfo import Workspace

workspace = Workspace(root=WORKSPACE)
print("Workspace root:", workspace.root)

## Step 1: Cell spec

In [ ]:
cell_spec = workspace.cell_spec(
    manufacturer="Samsung SDI",
    model="INR21700-50E",
    format="cylindrical",
    chemistry="Li-ion",
    positive_electrode_basis="NMC",
    negative_electrode_basis="graphite",
    specs={
        "nominal_capacity": {"value": 5.0,  "unit": "Ah"},
        "nominal_voltage":  {"value": 3.6,  "unit": "V"},
        "rated_energy":     {"value": 18.0, "unit": "Wh"},
        "mass":             {"value": 68.0, "unit": "g"},
    },
)

# IRIs are minted when the workspace is saved (Step 6).
print("Cell spec created:", cell_spec.manufacturer, cell_spec.model)

## Step 2: Cell instance

In [ ]:
cell = workspace.cell(
    cell_spec,
    serial_number="sdi-inr21700-50e-batch2026-A001",
)

print("Cell instance created:", cell.serial_number)
# The spec link is held as an object reference; the IRI pair is minted at save time (Step 6).
assert cell.cell_spec is cell_spec, "instance must reference the cell spec"
print("Link verified.")

## Step 3: Test protocol

In [ ]:
# PyBaMM experiment syntax: a list of step strings.
# BattINFO stores these verbatim so they can be passed directly to pybamm.Experiment(steps * cycles).
# Each step also receives an EMMO electrochemistry process type (ConstantCurrentDischarging,
# Resting, etc.) when it can be determined unambiguously from the step text.

PROTOCOL_STEPS = [
    "Discharge at 1C for 10 hours or until 2.5 V",
    "Rest for 10 minutes",
    "Charge at 1C until 4.2 V",
    "Hold at 4.2 V until C/20",
    "Rest for 10 minutes",
]

protocol = workspace.test_spec(
    name="1C Cycle Life at 25 °C",
    kind="cycling",
    steps=PROTOCOL_STEPS,
    cycles=3,
    description=(
        "1C CC-CV charge (4.2 V, C/20 cut-off) / 1C CC discharge (2.5 V cut-off). "
        "10-minute rest periods. 25 ± 2 °C. PyBaMM-compatible step syntax."
    ),
    version="1.0",
)

print("Test spec created:", protocol.name)
print("Steps (rendered back from the stored method):")
for i, step in enumerate(protocol.experiment, 1):
    print(f"  {i}. {step}")


### Semantic depth of a test protocol

The current representation types the protocol at the class level and carries a free-text
`description`. EMMO domain-electrochemistry has richer terms that allow a more explicit,
machine-replicable representation — here is what is available and when to use it.

#### What the ontology provides

| EMMO term | Meaning |
|---|---|
| `ConstantCurrentConstantVoltageCycling` | The complete CC-CV cycle-life protocol |
| `ConstantCurrentConstantVoltageCharging` | A single CC-CV charge step |
| `ConstantCurrentDischarging` | A single CC discharge step |
| `Resting` | A rest / OCV hold step |
| `ElectrochemicalTestingProcedure` | General electrochemical test container |
| `SerialWorkflow` / `IterativeWorkflow` | Structural containers for step sequences |
| `hasNext` / `precedes` / `follows` | Step-to-step sequencing predicates |
| `StepDuration`, `StepSignalCurrent`, `StepSignalVoltage` | Per-step parameters |

A 1C CC-CV cycle-life protocol could therefore be expressed as an `IterativeWorkflow`
containing a `SerialWorkflow` whose steps are:
`ConstantCurrentConstantVoltageCharging` → `Resting` → `ConstantCurrentDischarging` →
`Resting`, each step carrying typed parameters for C-rate, voltage limit, cut-off
current, rest duration, and temperature.

#### Pros of explicit step decomposition

- **Machine-replicable**: the test can be reproduced from the JSON-LD alone without
  parsing free text.
- **SPARQL-queryable at step level**: structured queries like *"find all datasets charged
  to 4.2 V at 1C CC-CV"* become precise rather than text-search-based.
- **Parameter validation**: step parameters (C-rate, voltage limits, temperatures) can
  be range-checked against cell specs and standards (IEC 62660, EUCAR) automatically.
- **Stable protocol identity**: the semantic structure survives even if the free-text
  description is incomplete or updated.

#### Cons and practical limits

- **Authoring burden**: a four-step cycle becomes a graph of ~10 typed nodes. No
  researcher will author this by hand, so tooling is a prerequisite.
- **EMMO coverage gaps**: GITT, interspersed EIS, multi-rate formation sequences, and
  temperature ramps do not yet have first-class EMMO step types. Mixed typed/free-text
  nodes undermine the value of the graph.
- **Cycler-file redundancy and false precision**: the ground-truth protocol is usually
  the binary file in Biologic BT-Lab, Maccor, or Arbin format. A parallel step graph
  that was hand-authored can silently diverge from what was actually run — which is
  worse than no graph at all.
- **Version fragility**: step-graph records couple tightly to specific EMMO term IRIs.
  An upstream rename invalidates stored records in a way that a text description does
  not.

#### Current recommendation

Type the protocol at the class level (e.g. `ConstantCurrentConstantVoltageCycling`) and
record protocol-level parameters (C-rate, voltage window, temperature, cut-off
condition) as typed properties. Keep the free-text `description` as the human-readable
ground truth. Reserve step-level decomposition for cases where:

1. The protocol is simple enough to express without gaps (standard CC, CC-CV, or rest
   steps only), **and**
2. The step graph is generated programmatically from a machine-readable protocol
   definition (e.g. a cycler script), not authored by hand.

When the cycler file is included as a dataset distribution, it already carries the
authoritative step definition — the EMMO class typing on the protocol record is
then sufficient for discovery and filtering.

## Step 4: Test

In [ ]:
test = workspace.test(
    cell,
    kind="cycling",
    protocol_ref=protocol,
    instrument="Biologic VMP-300",
    status="completed",
    description="1C cycle-life test on cell A001, April 2026.",
)

print("Test created:", test.description)

## Step 5: Dataset

In [ ]:
# Create a minimal stand-in CSV
data_dir = WORKSPACE / "inputs"
data_dir.mkdir(exist_ok=True)
csv_path = data_dir / "sdi-a001-cycle-life.csv"
csv_path.write_text(
    "cycle,capacity_ah,coulombic_efficiency\n"
    "1,4.98,0.934\n2,4.95,0.998\n3,4.93,0.998\n",
    encoding="utf-8",
)

dataset = workspace.dataset(
    cell,
    test=test,
    path=csv_path,
    title="INR21700-50E A001 cycle-life dataset",
    description="Voltage, current, and temperature time-series, 500 cycles.",
    license="CC-BY-4.0",
    curated_by="Jane Smith",
    comment="Cycle-life dataset from Guide 3 walkthrough.",
)

print("Dataset created:", dataset.name)

## Step 6: Save all records

In [ ]:
workspace.save()

# save() mints the canonical IRIs on the in-memory objects and links the chain.
print("Cell spec IRI:     ", cell_spec.id)
print("Cell instance IRI: ", cell.id)
print("Test spec IRI:     ", protocol.id)
print("Test IRI:          ", test.id)
print("Dataset IRI:       ", dataset.id)
assert cell.cell_spec_id == cell_spec.id, "cell_spec_id must match cell_spec.id"

print("\nRecords written:")
for record_file in sorted((WORKSPACE / "examples").rglob("*.json")):
    print(" ", record_file.relative_to(WORKSPACE))

### Export resolver JSON-LD for each record

`publish_record()` builds the resolver artifact for a single record: `index.json`, `index.jsonld`, and `index.html`. Running it on each saved record produces the full set of resolver artifacts for the chain.

In [ ]:
from battinfo.api import publish_record as _pr

for record_file in sorted((WORKSPACE / "examples").rglob("*.json")):
    _pr(record_file, target_root=WORKSPACE / "resolver")

resolver_jsonld_files = sorted((WORKSPACE / "resolver").rglob("index.jsonld"))
print(f"Resolver JSON-LD files written ({len(resolver_jsonld_files)}):")
for f in resolver_jsonld_files:
    print(f"  {f.relative_to(WORKSPACE)}")

## Step 7: Validate the chain

In [ ]:
from battinfo.validate.record import validate_record_report

# validate_record_report takes a dict, not a path
dataset_path = next((WORKSPACE / "examples" / "dataset").glob("*.json"), None)
if dataset_path:
    dataset_doc = json.loads(dataset_path.read_text(encoding="utf-8"))
    report = validate_record_report(dataset_doc, source_root=WORKSPACE / "examples")
    print("ok:", report.ok, "| errors:", len(report.errors), "| warnings:", len(report.warnings))
else:
    print("No dataset record found — run workspace.save() first.")

## Step 8: Build a publication package

In [ ]:
package = workspace.build_publication_package(dataset)
pkg_path = Path(package["publish_path"])

print("Package:", pkg_path.relative_to(WORKSPACE))

bundle = json.loads(pkg_path.read_text(encoding="utf-8"))
print(f"Nodes in @graph: {len(bundle['@graph'])}")
for node in bundle["@graph"]:
    types = node.get("@type", "?")
    uid = node.get("@id", "").split("/")[-1]
    print(f"  {types}  —  {uid}")

### Full publication package JSON-LD

The publication package is a single JSON-LD document with a `@graph` containing all four linked records. This is the artifact submitted to a registry or shared with collaborators.

In [ ]:
print(json.dumps(bundle, indent=2))

## Step 9: Publish

In [ ]:
from battinfo import publish

result = publish(cell_spec, destination="local", root=WORKSPACE / "published")
print("Published IRI:", result.canonical_iri)

# Registry publish (fill in credentials):
# result = publish(
#     cell_spec, destination="registry",
#     root=WORKSPACE / "submissions",
#     registry_base_url="https://registry.example.org",
#     api_key="your-api-key",
#     workspace_id="example-lab",
#     publisher_id="sintef-industry",
#     source_version="2026-05-04",
# )
print("Registry publish shown commented out.")

## Querying saved records

In [ ]:
from battinfo import query_cell_specs, query_datasets

cell_specs = query_cell_specs(
    chemistry="Li-ion",
    format="cylindrical",
    cell_specs_dir=WORKSPACE / "examples" / "cell-spec",
)
print(f"Found {len(cell_specs)} cylindrical Li-ion cell spec(s)")

datasets = query_datasets(
    related_cell_id=cell.id,
    directory=WORKSPACE / "examples" / "dataset",
)
print(f"Found {len(datasets)} dataset(s) linked to cell {cell.id.split('/')[-1]}")

## Loading a received package

In [ ]:
from battinfo import load_publication_package

bundle_obj = load_publication_package(pkg_path)

print("Cell spec manufacturer:", bundle_obj.cell_spec.manufacturer)
print("Cell serial number:    ", bundle_obj.cell_instance.serial_number)
print("Test kind:             ", bundle_obj.test.test_kind)
print("Dataset object type:   ", type(bundle_obj.dataset).__name__)

## Next

- **[Guide 4 — Semantic layer](04-semantic-layer.ipynb)**: understand the JSON-LD and work with it as RDF
- **[Guide 5 — Descriptors](05-descriptors.ipynb)**: add electrode composition and construction details